# Post process data extracted using LLMs


In [53]:
import numpy as np
import pandas as pd
import geopy as gpy
import time
from src.data import *


In [4]:
#load data
fnout = "5labelled_response_test1.csv"
response_df = pd.read_csv(DATA_OUT_PATH+fnout)

In [5]:
response_df

,Unnamed: 0,hazardType,country,region,state,city,locationAnnotation,startYear,startMonth,startDay,endYear,endMonth,endDay,hazardName,hazardSubtypes,appealCode,location,date,disasterType
0,0,Flood,Algeria,Southern and Western Algeria,"Béchar, Tamanrasset, El Bayadh, Tiaret, Tindou...","Béchar, Elbayadh, Beni Abbes, Tamanrasset","The most affected areas include Béchar, Elbaya...",2024,9,5,2024,9,8,NaN,[],MDRDZ011,Algeria,22/09/2024,Famine / Food Insecurity
1,1,Storm,Algeria,Southern and Western Algeria,"Béchar, Tamanrasset, El Bayadh, Tiaret, Tindou...","Béchar, Elbayadh, Beni Abbes, Tamanrasset","On September 8, 2024, a severe tropical distur...",2024,9,5,2024,9,8,NaN,"[""tropical storm""]",MDRDZ011,Algeria,22/09/2024,Famine / Food Insecurity
2,2,Mass movement,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa...",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Areas such as Jacobabad, Naushahro Feroz, Ghot...",2024,7,1,2024,9,1,NaN,"[""landslide"", ""mudslide""]",MDRPK026,Pakistan,17/09/2024,Flood
3,3,Flood,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Areas such as Jacobabad, Naushahro Feroz, Ghot...",2024,7,1,2024,9,1,NaN,"[""flash flood"", ""riverine flood""]",MDRPK026,Pakistan,17/09/2024,Flood
4,4,Extreme temperature,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa...",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Regionally, Balochistan received 239 per cent ...",2024,7,1,2024,9,1,NaN,[],MDRPK026,Pakistan,17/09/2024,Flood
5,5,Flood,Cameroon,Extrême-Nord,NaN,"Yagoua, Blangoua, Mackary, Zina, Maga",Cameroon's Far North region has been experienc...,2024,7,1,2024,8,28,Flood,[],MDRCM039,Cameroon,13/09/2024,Flood
6,6,Flood,Benin,Couffo,NaN,NaN,Intense rainfall observed in the departments o...,2024,6,26,2024,6,26,NaN,"[""riverine flood""]",MDRBJ019,Benin,07/09/2024,Flood
7,7,Mass movement,Sudan,NaN,"Red Sea, River Nile, Northern State",NaN,"So far, Red Sea, River Nile, and Northern Stat...",2024,6,1,2024,8,12,NaN,[],MDRSD034,Sudan,06/09/2024,Flood
8,8,Flood,Sudan,NaN,"Red Sea, River Nile, Northern State",NaN,"So far, Red Sea, River Nile, and Northern Stat...",2024,6,1,2024,8,12,NaN,"[""flash flood"", ""riverine flood""]",MDRSD034,Sudan,06/09/2024,Flood


In [ ]:
#add iso3
from src.LLM_functions import country_name_to_iso3
response_df["country_iso3"] = response_df["country"].apply(country_name_to_iso3)

In [ ]:
def separate_locs(locations):
    return locations.split(",")

In [28]:
geolocator = gpy.geocoders.Nominatim(user_agent='luca.severino@usys.ethz.ch')
response_row = response_df.iloc[1]
country = response_row["country"]
regions = response_row["region"].split(",")
cities = response_row["city"].split(",")
queries = {}
i = 0
for ctry, reg, city in zip([country]*len(regions), regions, cities):
    nominatim_query = {
        "country": ctry,
        "region": reg,
        "city": city
    }
    nominatim_result = geolocator.geocode(nominatim_query)
    if nominatim_result is None:
        print("No results for query: ", nominatim_query)
        continue
    else:
        nominatim_query["latitude"] = nominatim_result.latitude
        nominatim_query["longitude"] = nominatim_result.longitude
        queries[i] = nominatim_query
    i+=1


In [ ]:
#geolocate location text using nominatim
def make_nominatim_query(response_row, geolocator):
    '''Function to extract geoloactions from text using nominatim'''
    country = response_row["country"]
    regions = response_row["region"].split(",")
    cities = response_row["city"].split(",")
    df_list = []
    i = 0
    time_last_request = time.time()
    for ctry, reg, city in zip([country]*len(regions), regions, cities):
        nominatim_query = {
            "country": ctry,
            "region": reg,
            "city": city
        }
        time_new_request = time.time()
        if time_new_request - time_last_request < 1:
            time.sleep(time_new_request - time_last_request)
        nominatim_result = geolocator.geocode(nominatim_query)
        time_last_request = time_new_request
        if nominatim_result is None:
            print("No results for query: ", nominatim_query)
            continue
        else:
            nominatim_query["latitude"] = nominatim_result.latitude
            nominatim_query["longitude"] = nominatim_result.longitude
            df_list.append(pd.DataFrame(nominatim_query, index=pd.MultiIndex.from_tuples([(response_row.appealCode, i)], names=['appealCode', 'index'])))
        i+=1


    return df_list

In [59]:
list_all_locs = []
for i, row in response_df.iterrows():
    loc_list = make_nominatim_query(row, geolocator)
    if loc_list:
        list_all_locs.extend(loc_list)
    else:
        print(f"No results for row: {i}, appealCode: {row['appealCode']}")
df_all_locs = pd.concat(list_all_locs)

No results for query:  {'country': 'Pakistan', 'region': ' Sindh', 'city': ' Naushahro Feroz'}
No results for query:  {'country': 'Pakistan', 'region': ' Sindh', 'city': ' Naushahro Feroz'}
No results for query:  {'country': 'Pakistan', 'region': ' Sindh', 'city': ' Naushahro Feroz'}


AttributeError: 'float' object has no attribute 'split'

In [48]:
response_df

,Unnamed: 0,hazardType,country,region,state,city,locationAnnotation,startYear,startMonth,startDay,endYear,endMonth,endDay,hazardName,hazardSubtypes,appealCode,location,date,disasterType,country_iso3
0,0,Flood,Algeria,Southern and Western Algeria,"Béchar, Tamanrasset, El Bayadh, Tiaret, Tindou...","Béchar, Elbayadh, Beni Abbes, Tamanrasset","The most affected areas include Béchar, Elbaya...",2024,9,5,2024,9,8,NaN,[],MDRDZ011,Algeria,22/09/2024,Famine / Food Insecurity,DZA
1,1,Storm,Algeria,Southern and Western Algeria,"Béchar, Tamanrasset, El Bayadh, Tiaret, Tindou...","Béchar, Elbayadh, Beni Abbes, Tamanrasset","On September 8, 2024, a severe tropical distur...",2024,9,5,2024,9,8,NaN,"[""tropical storm""]",MDRDZ011,Algeria,22/09/2024,Famine / Food Insecurity,DZA
2,2,Mass movement,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa...",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Areas such as Jacobabad, Naushahro Feroz, Ghot...",2024,7,1,2024,9,1,NaN,"[""landslide"", ""mudslide""]",MDRPK026,Pakistan,17/09/2024,Flood,PAK
3,3,Flood,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Areas such as Jacobabad, Naushahro Feroz, Ghot...",2024,7,1,2024,9,1,NaN,"[""flash flood"", ""riverine flood""]",MDRPK026,Pakistan,17/09/2024,Flood,PAK
4,4,Extreme temperature,Pakistan,"Balochistan, Sindh, Punjab, Khyber Pakhtunkhwa...",NaN,"Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sa...","Regionally, Balochistan received 239 per cent ...",2024,7,1,2024,9,1,NaN,[],MDRPK026,Pakistan,17/09/2024,Flood,PAK
5,5,Flood,Cameroon,Extrême-Nord,NaN,"Yagoua, Blangoua, Mackary, Zina, Maga",Cameroon's Far North region has been experienc...,2024,7,1,2024,8,28,Flood,[],MDRCM039,Cameroon,13/09/2024,Flood,CMR
6,6,Flood,Benin,Couffo,NaN,NaN,Intense rainfall observed in the departments o...,2024,6,26,2024,6,26,NaN,"[""riverine flood""]",MDRBJ019,Benin,07/09/2024,Flood,BEN
7,7,Mass movement,Sudan,NaN,"Red Sea, River Nile, Northern State",NaN,"So far, Red Sea, River Nile, and Northern Stat...",2024,6,1,2024,8,12,NaN,[],MDRSD034,Sudan,06/09/2024,Flood,SDN
8,8,Flood,Sudan,NaN,"Red Sea, River Nile, Northern State",NaN,"So far, Red Sea, River Nile, and Northern Stat...",2024,6,1,2024,8,12,NaN,"[""flash flood"", ""riverine flood""]",MDRSD034,Sudan,06/09/2024,Flood,SDN
